# ローカルLLMワークショップ

このノートブックでは、**Ollama** を使ってローカルLLMをGoogle Colab上で動かし、日本語での推論性能やエージェント機能を体験します。

## アジェンダ

1. LLM とは
2. ローカル LLM
3. LLMハンズオン
    - セットアップ（Ollama インストール・サーバー起動・モデルダウンロード）
    - 基本的な推論
    - システムプロンプト
    - マルチターン会話
    - ストリーミング
    - 構造化出力（JSON）
    - ツール呼び出し（Tool Use）
    - パラメータ調整
    - エージェントワークフロー
4. まとめ

# 1. LLM とは

## 概要

LLM（Large Language Model）は、大量のテキストデータを学習した大規模な言語モデルです。

| 時期 | 出来事 |
|---|---|
| 2017年 | Google が Transformer アーキテクチャを発表（"Attention is All You Need"）|
| 2020年 | OpenAI が GPT-3 を公開。スケーリング則により性能が劇的に向上 |
| 2022年 | ChatGPT の登場により LLM が一般に普及 |
| 2023年〜 | Meta の Llama をはじめとするオープンソース LLM が急速に発展 |
| 2024年〜 | Qwen、Gemma、Phi など小型でも高性能な SLM（Small Language Model）が登場 |

## 仕組み

LLM は **Transformer** のアーキテクチャをベースにしています。

1. **トークン化**: テキストを「トークン」という単位に分割する
2. **Attention 機構**: 各トークンが他のトークンとどれだけ関連しているかを計算する（文脈の理解）
3. **次トークン予測**: これまでのトークン列を元に、次に来るトークンの確率分布を計算する
4. **サンプリング**: 確率分布からトークンを選択して出力する（`temperature` などで制御）

```
入力: "東京は日本の"
        ↓ トークン化
["東京", "は", "日本", "の"]
        ↓ Attention で文脈を計算
        ↓ 次トークンの確率分布
{"首都": 0.85, "都市": 0.08, "中心": 0.04, ...}
        ↓ サンプリング
出力: "首都"
```

> 💡 LLM は「確率的に尤もらしいトークン」を選び続けることで文章を生成します。正確な事実を記憶しているわけではないため、ハルシネーション（事実と異なる出力）が発生することがあります。

# 2. ローカル LLM

## 導入

これまで LLM はクラウド API（OpenAI、Anthropic など）経由での利用が主流でしたが、近年はエッジデバイスでローカル実行する動きが広がっています。

**ローカル LLM のメリット:**
- **プライバシー**: データが外部サーバーに送信されない
- **コスト**: API 利用料が不要
- **レイテンシ**: ネットワーク遅延がない
- **オフライン動作**: インターネット接続不要

**活用事例:**
- MacBook Pro（Apple Silicon）での日常利用
- Jetson Orin など組み込みデバイスでのエッジ推論
- 社内データを扱うオンプレミス環境

## 代表的なモデル

| モデル | 開発元 | 特徴 |
|---|---|---|
| **Llama 3.2** | Meta | オープンソースの代表格。多言語対応 |
| **Qwen3** | Alibaba | 日本語性能が高い。Thinking モード搭載 |
| **Gemma 3** | Google | 軽量で高性能。商用利用可能 |
| **Phi-4** | Microsoft | 小型ながら推論性能が高い |
| **Mistral** | Mistral AI | ヨーロッパ発。コード生成が得意 |

## 基礎知識

### パラメータ数

モデルの「重み」の数。多いほど性能が高い傾向があるが、必要なメモリ量も増える。
Q4 量子化の場合、おおよそ **1B パラメータ ≈ 0.6GB** のメモリが目安です。

| パラメータ数 | 必要メモリ（Q4目安） | 動作可能な主なデバイス |
|---|---|---|
| 1B〜3B | 1〜2GB | スマートフォン、Raspberry Pi |
| 4B〜7B | 2.5〜4.5GB | Jetson Orin Nano（8GB）、一般的な PC |
| 8B〜14B | 5〜9GB | Google Colab T4（15GB VRAM）、RTX 3080 |
| 14B〜32B | 9〜20GB | MacBook Pro（24GB 統合メモリ）、RTX 4090（24GB）|
| 32B〜70B | 20〜45GB | RTX 5090（32GB VRAM）、A100（40GB） |
| 70B〜 | 45GB〜 | マルチ GPU、H100 サーバー |

#### 代表的なデバイスと搭載メモリ

| デバイス | メモリ | 特徴 | 動かせるモデルの目安 |
|---|---|---|---|
| **Jetson Orin Nano** | 8GB（CPU/GPU 共有） | エッジ AI 組み込みボード | 〜4B（Q4） |
| **Google Colab T4** | 15GB VRAM | 今回使用するクラウド GPU | 〜8B（Q4） |
| **MacBook Pro M3/M4** | 24GB（統合メモリ） | CPU・GPU・RAM が共有のため効率的 | 〜14B（Q4） |
| **RTX 4090** | 24GB VRAM | デスクトップ向けハイエンド GPU | 〜14B（Q4） |
| **RTX 5090** | 32GB VRAM | 2025年時点の最高峰コンシューマ GPU | 〜32B（Q4） |
| **A100** | 40〜80GB VRAM | データセンター向け GPU | 〜70B（Q4） |

> 💡 **Apple Silicon の強み**: MacBook Pro の統合メモリ（Unified Memory）は CPU と GPU が共有するため、24GB すべてをモデルに割り当てられます。同じ 24GB でも VRAM 専用の GPU より大きなモデルを扱えます。

### 量子化

浮動小数点の精度を下げてモデルサイズを圧縮する技術。
- `FP16`（16bit）→ `Q8`（8bit）→ `Q4`（4bit） と下げるにつれ、メモリは削減されるが精度もわずかに低下する。
- Ollama はデフォルトで量子化済みモデル（主に Q4）を使用する。

### モデルファイル形式

ローカル LLM のモデルファイルにはいくつかの形式があります。用途や推論エンジンに合わせて使い分けます。

| 形式 | 特徴 | 主な用途 |
|---|---|---|
| **GGUF** | llama.cpp 向けの標準形式。量子化を内包し CPU/GPU 両対応 | Ollama・LM Studio・llama.cpp |
| **Safetensors** | HuggingFace が推進する安全な形式。悪意あるコードを含められない | Python（Transformers）での学習・推論 |
| **GGML** | GGUF の前身。現在は非推奨 | 旧来の llama.cpp |
| **AWQ / GPTQ** | GPU 向け量子化形式。vLLM や Transformers で使用 | GPU サーバーでの高速推論 |

**GGUF が重要な理由:**
- Ollama が内部で使用しているファイル形式が GGUF
- 量子化レベル（Q4_K_M、Q8_0 など）をファイル名に含むため、精度とサイズのトレードオフが一目でわかる
- 1ファイルにモデル全体が収まるため配布・管理が容易

```
# GGUF ファイル名の例
qwen3-4b-q4_k_m.gguf   ← Q4量子化（K-means, Medium）/ 約2.5GB
qwen3-4b-q8_0.gguf     ← Q8量子化 / 約4.5GB
qwen3-4b-f16.gguf      ← FP16（無量子化）/ 約8GB
```

### Dense vs MoE

| | Dense | MoE（Mixture of Experts）|
|---|---|---|
| **仕組み** | 全パラメータを毎回使用 | 入力ごとに一部の「専門家」のみ使用 |
| **メモリ効率** | パラメータ数に比例 | 総パラメータ数は大きいが推論時の使用量は少ない |
| **性能** | パラメータ数に依存 | 同じ計算コストでより高い性能を実現しやすい |
| **主な用途** | ローカル LLM | クラウド・企業運用モデル |
| **代表例** | Qwen3-4B、Llama 3.2、Gemma 3 | GPT-4、Claude、Gemini、Grok |

**クラウド・企業モデルは MoE が標準になりつつある:**
OpenAI の GPT-4、Google の Gemini、xAI の Grok など、現在主流の商用モデルの多くは MoE アーキテクチャを採用しています。数百B〜数兆のパラメータを持ちながら、推論時は一部のみ使用するため、コストと性能を両立しています。

**ローカル LLM は Dense モデルが中心:**
GGUF で配布されるローカル向けモデルの多くは Dense モデルです。MoE より軽量で動かしやすい反面、同じメモリ量でのモデルサイズには上限があり、知識量・推論深度ではクラウドモデルに劣りやすい傾向があります。

> 💡 ローカル LLM とクラウド LLM は「同じ技術だが性能差がある」のではなく、アーキテクチャ自体が異なります。用途に応じて使い分けることが重要です。

### Thinking モード / No-Thinking モード

Qwen3 や DeepSeek-R1 などの一部モデルは、回答前に内部で「思考」を行う **Thinking モード** を搭載しています。

#### Thinking モード（`think=True`）
- 回答前に `<think>...</think>` ブロック内で段階的に推論を行う
- 数学・論理・コーディングなど複雑な問題で精度が向上する
- トークン消費が多く、応答時間が長くなる

#### No-Thinking モード（`think=False`）
- 思考プロセスをスキップして直接回答する
- 応答が速く、トークン消費が少ない
- 日常的な質問・チャット・情報検索など単純なタスクに向いている

```python
# Ollama SDK での切り替え
response = ollama.chat(model="qwen3.5:4b", messages=messages, think=True)   # 深い推論
response = ollama.chat(model="qwen3.5:4b", messages=messages, think=False)  # 高速応答
```

| | Thinking | No-Thinking |
|---|---|---|
| **精度** | 高い（複雑な問題） | 標準 |
| **速度** | 遅い | 速い |
| **トークン消費** | 多い | 少ない |
| **向いているタスク** | 数学・推論・コード生成 | 会話・翻訳・情報検索 |

> 💡 このワークショップでは応答速度を優先するため `think=False` を使用しています。

## 推論エンジン

| エンジン | 特徴 | 向いている用途 |
|---|---|---|
| **Ollama** | セットアップが簡単。API サーバー機能付き | 開発・プロトタイピング |
| **llama.cpp** | C++ 実装で超軽量。CPU でも動作 | 組み込み・低スペック環境 |
| **vLLM** | 高スループット。本番サービス向け | 大規模サービス |
| **LM Studio** | GUI 付き。初心者向け | 個人利用・デモ |

このワークショップでは **Ollama** を使用します。

# 3. LLMハンズオン

ここからは実際にコードを動かしながら LLM の機能を体験します。
各セルを上から順番に実行してください。

## セットアップ

まず GPU が使用可能か確認し、Ollama をインストールします。

### GPU の確認

In [ ]:
!nvidia-smi

In [ ]:
# 解凍ツール zstd をインストール（Ollama のインストールスクリプトが必要とする）
!apt-get install -y zstd

# Ollama をインストール（ローカル LLM の推論エンジン）
!curl -fsSL https://ollama.com/install.sh | sh

# Ollama Python SDK をインストール
%pip install -q ollama

### Ollama サーバーの起動とモデルのダウンロード

インストール完了後、Ollama サーバーをバックグラウンドで起動し、使用するモデルをダウンロードします。

In [ ]:
import json
import subprocess
import time
import urllib.request

import requests
import ollama

# 使用するモデル名（変更する場合はここだけ修正する）
# MODEL = "qwen3.5:0.8b"
MODEL = "qwen3.5:4b"
# MODEL = "qwen3.5:9b"
# MODEL = "gemma4:e2b"
# MODEL = "gemma4:e4b"

In [ ]:
# Ollama サーバーをバックグラウンドで起動
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# サーバーが起動するまで待機（最大15秒）
for _ in range(15):
    try:
        urllib.request.urlopen("http://localhost:11434")
        print("Ollama サーバーが起動しました")
        break
    except Exception:
        time.sleep(1)
else:
    print("エラー: Ollama サーバーが起動しませんでした。このセルを再実行してください。")

In [ ]:
!ollama pull {MODEL}

## 2. 基本的な推論

Ollama SDK を使って LLM に質問してみましょう。`ollama.chat()` に `model` と `messages` を渡すだけで推論できます。

- `think=False` を指定することで、Qwen3 の内部思考（reasoning）をスキップし、直接回答を返します。

In [ ]:
# 基本的な推論: 英語で質問
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "What is the capital of Japan?"}],
    think=False,  # 思考モードをオフにして直接回答
)

print(response.message.content)

In [ ]:
# 基本的な推論: 日本語で質問
response = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": "日本の首都はどこですか？"}],
    think=False,
)

print(response.message.content)

## 3. システムプロンプト

`system` ロールのメッセージを先頭に追加することで、モデルの振る舞いや回答スタイルを制御できます。

以下の例では、英語で質問しても日本語で回答するよう指示しています。

In [ ]:
# システムプロンプトで「日本語で回答する」よう指示し、英語で質問する
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user",   "content": "What is the capital of France?"},
    ],
    think=False,
)

print(response.message.content)

## 4. マルチターン会話

会話履歴（`messages` リスト）にやり取りを追加し続けることで、モデルは文脈を保持したまま会話できます。

アシスタントの返答を `role: assistant` として履歴に追加するのがポイントです。

In [ ]:
# 会話履歴を保持するリスト（システムプロンプトを先頭に追加）
messages = [
    {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
]

# 複数ターンの発話リスト
turns = [
    "私の名前は太郎です。",
    "私の名前は何ですか？",       # 1ターン前の情報を参照できるか確認
    "私の名前をローマ字で書いてください。",
]

for user_input in turns:
    # ユーザーの発話を履歴に追加
    messages.append({"role": "user", "content": user_input})
    print(f"User: {user_input}")

    response = ollama.chat(model=MODEL, messages=messages, think=False)

    # アシスタントの返答を履歴に追加して文脈を維持
    assistant_reply = response.message.content
    messages.append({"role": "assistant", "content": assistant_reply})
    print(f"Assistant: {assistant_reply}\n")

## 5. ストリーミング

`stream=True` を指定すると、生成されたトークンをリアルタイムで受け取れます。
長い回答でも最初の文字からすぐに表示されるため、UX が向上します。

In [ ]:
# stream=True でトークンをリアルタイムに受け取る
stream = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "あなたは親切な日本語アシスタントです。必ず日本語で答えてください。"},
        {"role": "user",   "content": "日本の四季について説明してください。"},
    ],
    stream=True,
    think=False,
)

# 生成されたトークンを順次出力（end='' で改行せず flush=True で即時表示）
for chunk in stream:
    print(chunk.message.content, end="", flush=True)

## 6. 構造化出力（JSON）

`format="json"` を指定すると、モデルは必ず有効な JSON を返します。
データ抽出やシステム連携など、出力を確実にパースしたい場面で活用します。

In [ ]:
# format="json" で必ず有効な JSON を返すよう強制する
response = ollama.chat(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant that responds only in valid JSON."},
        {"role": "user", "content": (
            "日本の主要都市を3つ挙げて、以下のJSON形式で返してください:\n"
            '{"cities": [{"name": "...", "prefecture": "...", "population": ...}]}'
        )},
    ],
    format="json",
    think=False,
)

# JSON 文字列をパースして整形表示
result = json.loads(response.message.content)
print(json.dumps(result, ensure_ascii=False, indent=2))

## 7. ツール呼び出し（Tool Use）

ツール定義を渡すと、モデルは必要に応じて外部関数の呼び出しを指示します。
実際の関数はPython側で実行し、結果を `role: tool` として返すことで、最終回答を生成します。

以下の例では、**Open-Meteo API**（無料・APIキー不要）を使って実際の天気情報を取得します。

In [ ]:
# ツール定義: モデルが呼び出せる関数のスキーマを定義する
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "指定した都市の現在の天気を取得する",
            "parameters": {
                "type": "object",
                "properties": {
                    # モデルは英語の都市名を渡す（ジオコーディング API の仕様）
                    "city": {"type": "string", "description": "City name in English (e.g. Tokyo, Osaka)"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度の単位"},
                },
                "required": ["city"],
            },
        },
    }
]

def get_weather(city, unit="celsius"):
    # ステップ1: 都市名を緯度・経度に変換（Open-Meteo ジオコーディング API）
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1}
    ).json()
    if not geo.get("results"):
        return json.dumps({"error": f"都市 '{city}' が見つかりませんでした。"}, ensure_ascii=False)
    location = geo["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    # ステップ2: 座標から現在の天気を取得（Open-Meteo 気象 API）
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": lat, "longitude": lon,
            "current": "temperature_2m,weathercode,windspeed_10m",
            "temperature_unit": unit,
        }
    ).json()
    current = weather["current"]
    return json.dumps({
        "city": city, "temperature": current["temperature_2m"],
        "unit": unit, "windspeed_kmh": current["windspeed_10m"],
        "weathercode": current["weathercode"],
    }, ensure_ascii=False)

# ステップ1: ツールを渡してモデルに質問（モデルがツール呼び出しを判断する）
messages = [{"role": "user", "content": "東京の天気を教えてください。"}]
response = ollama.chat(model=MODEL, messages=messages, tools=tools, think=False)

messages.append(response.message)

# ステップ2: モデルがツール呼び出しを要求した場合は実行する
if response.message.tool_calls:
    for tool_call in response.message.tool_calls:
        args = tool_call.function.arguments  # Ollama SDK では引数は dict で返る
        print(f"ツール呼び出し: {tool_call.function.name}({args})")
        result = get_weather(**args)
        print(f"ツール結果: {result}\n")
        messages.append({"role": "tool", "content": result})

    # ステップ3: ツールの実行結果を渡して最終回答を生成
    final_response = ollama.chat(model=MODEL, messages=messages, think=False)
    print(f"アシスタント: {final_response.message.content}")
else:
    print(f"アシスタント: {response.message.content}")

## 8. パラメータ調整

推論の挙動は以下のパラメータで制御できます。`options={}` に辞書形式で渡します。

| パラメータ | 説明 | 推奨範囲 |
|---|---|---|
| `temperature` | 出力のランダム性。低いほど一貫性が高く、高いほど創造的 | 0.0 〜 2.0 |
| `top_p` | 確率上位 X% のトークンのみからサンプリング（nucleus sampling） | 0.0 〜 1.0 |
| `frequency_penalty` | 既出トークンの繰り返しを抑制 | -2.0 〜 2.0 |

同じプロンプトで各設定の出力を比較してみましょう。

In [ ]:
# 同じプロンプトでパラメータを変えて出力を比較する
messages = [
    {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations, no titles, no romanization."},
    {"role": "user",   "content": "俳句を一句詠んでください。"},
]

# 各設定の label と options を定義
configs = [
    {"label": "temperature=0.0 (決定的・一貫性重視)",  "options": {"temperature": 0.0}},
    {"label": "temperature=1.0 (バランス・デフォルト)", "options": {"temperature": 1.0}},
    {"label": "temperature=2.0 (ランダム・創造的)",    "options": {"temperature": 2.0}},
    {"label": "top_p=0.1 (保守的な候補に絞る)",        "options": {"temperature": 1.0, "top_p": 0.1}},
    {"label": "frequency_penalty=2.0 (繰り返し抑制)",  "options": {"temperature": 1.0, "frequency_penalty": 2.0}},
]

for config in configs:
    label = config.pop("label")  # label は API パラメータではないため取り出す
    response = ollama.chat(
        model=MODEL,
        messages=messages,
        think=False,
        **config,
    )
    print(f"[{label}]")
    print(response.message.content.strip())
    print()

## 9. エージェントワークフロー

複数の LLM を組み合わせて、役割分担したエージェントパイプラインを構築できます。

```
ユーザー入力
    ↓
[ルーターAgent]   → 入力を分類（weather / haiku / general）
    ↓
[スペシャリストAgent]
  • weather  → Tool Use で天気APIを呼び出し
  • haiku    → 俳句を生成
  • general  → 一般的な質問に回答
    ↓
[ガードレールAgent] → 回答を丁寧な日本語に書き換え
```

In [ ]:
# ── 1. ルーターAgent ───────────────────────────────────────────────────────────
def router_agent(user_input: str) -> str:
    # ユーザー入力を分類してスペシャリストのラベルを返す
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                'You are a routing agent. Classify the user input into exactly one label. '
                'Reply ONLY with valid JSON: {"label": "<label>"} '
                'Labels: "weather" (weather questions), "haiku" (creative/poem requests), "general" (everything else).'
            )},
            {"role": "user", "content": user_input},
        ],
        format="json",  # 必ず JSON で返す
        think=False,
    )
    result = json.loads(response.message.content)
    print(f"[ルーター] → ラベル: {result['label']}")
    return result["label"]

# ── 2. スペシャリストAgent ─────────────────────────────────────────────────────
def weather_specialist(user_input: str) -> str:
    # 天気専門Agent: Tool Use で Open-Meteo API を呼び出して実際の天気を返す
    tools = [{
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name in English"},
                },
                "required": ["city"],
            },
        },
    }]

    def get_weather(city):
        # 都市名を緯度経度に変換
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city, "count": 1}).json()
        if not geo.get("results"):
            return json.dumps({"error": f"City '{city}' not found."})
        loc = geo["results"][0]
        # 緯度経度から現在の天気を取得
        weather = requests.get("https://api.open-meteo.com/v1/forecast", params={
            "latitude": loc["latitude"], "longitude": loc["longitude"],
            "current": "temperature_2m,weathercode,windspeed_10m",
        }).json()
        c = weather["current"]
        return json.dumps({"city": city, "temperature_celsius": c["temperature_2m"],
                           "windspeed_kmh": c["windspeed_10m"]})

    messages = [{"role": "user", "content": user_input}]
    res = ollama.chat(model=MODEL, messages=messages, tools=tools, think=False)
    messages.append(res.message)

    if res.message.tool_calls:
        for tc in res.message.tool_calls:
            args = tc.function.arguments  # Ollama SDK では引数は dict で返る
            tool_result = get_weather(**args)
            print(f"[天気スペシャリスト] ツール呼び出し: get_weather({args})")
            messages.append({"role": "tool", "content": tool_result})
        # ツール結果を渡して最終回答を生成
        final = ollama.chat(model=MODEL, messages=messages, think=False)
        return final.message.content
    return res.message.content

def haiku_specialist(user_input: str) -> str:
    # 俳句専門Agent: 3行の俳句のみを返す
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a haiku poet. Reply with exactly one haiku (3 lines only). No explanations."},
            {"role": "user", "content": user_input},
        ],
        think=False,
    )
    return res.message.content

def general_specialist(user_input: str) -> str:
    # 汎用Agent: 日本語で簡潔に回答する
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer concisely in Japanese."},
            {"role": "user", "content": user_input},
        ],
        think=False,
    )
    return res.message.content

# ラベルとスペシャリスト関数のマッピング
SPECIALISTS = {
    "weather": weather_specialist,
    "haiku":   haiku_specialist,
    "general": general_specialist,
}

# ── 3. ガードレールAgent ───────────────────────────────────────────────────────
def guardrail_agent(specialist_output: str) -> str:
    # スペシャリストの出力を丁寧な日本語に書き換える
    res = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": (
                "You are a politeness guardrail. Rewrite the given response to be warm, "
                "polite, and natural in Japanese. Keep the content accurate. Output the rewritten response only."
            )},
            {"role": "user", "content": specialist_output},
        ],
        think=False,
    )
    return res.message.content

# ── パイプラインの実行 ──────────────────────────────────────────────────────────
def run_agent(user_input: str):
    print(f"ユーザー: {user_input}\n")
    label    = router_agent(user_input)
    raw      = SPECIALISTS.get(label, general_specialist)(user_input)
    print(f"[スペシャリスト ({label})]\n{raw}\n")
    polished = guardrail_agent(raw)
    print(f"[ガードレール] 最終回答:\n{polished}")

run_agent("東京の天気を教えてください。")
# run_agent("秋について俳句を詠んでください。")
# run_agent("日本で一番高い山は何ですか？")

# 4. まとめ

## 本日学んだこと

| セクション | 内容 |
|---|---|
| LLM の基礎 | Transformer・Attention 機構・トークン予測の仕組み |
| ローカル LLM | Ollama によるセットアップと Qwen3.5:4B の動作確認 |
| 基本推論 | `ollama.chat()` を使った日本語推論 |
| システムプロンプト | モデルの振る舞いをシステムメッセージで制御 |
| マルチターン | 会話履歴を渡して文脈を保持 |
| ストリーミング | `stream=True` でリアルタイム出力 |
| 構造化出力 | `format="json"` で確実な JSON 取得 |
| ツール呼び出し | 外部 API と連携したリアルタイム情報取得 |
| パラメータ調整 | `temperature`・`top_p`・`frequency_penalty` の効果 |
| エージェントワークフロー | ルーター・スペシャリスト・ガードレールの 3 層構成 |

## 発展トピック

### ファインチューニング（Fine-tuning）
特定ドメインのデータでモデルを追加学習させ、専門性を高める。
**[Unsloth](https://github.com/unslothai/unsloth)** を使うと、少ない VRAM でも効率的にファインチューニングができる。

### RAG（Retrieval-Augmented Generation）
社内ドキュメントや最新情報をベクトルDBに格納し、質問に関連する情報を検索してプロンプトに付与する手法。
ハルシネーションを抑えつつ、最新・専門情報に対応できる。

### エージェントフレームワーク
**[LangGraph](https://github.com/langchain-ai/langgraph)** を使うと、ノード・エッジでエージェントの状態遷移を管理できる。
複雑なマルチエージェントシステムの構築に適している。

### LLM の仕組みをさらに深く学ぶ
Andrej Karpathy の **[nanoGPT](https://github.com/karpathy/nanoGPT)** / **[minGPT](https://github.com/karpathy/minGPT)** は、GPT をゼロから実装したシンプルなコードで、Transformer の仕組みを理解するのに最適です。

お疲れ様でした！質問があればお気軽にどうぞ 🎉